In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l5.3 Learning-rate schedules

> 🎯 **Goal:** See why the learning rate should change over training, and build the warmup-then-cosine-decay schedule that covers almost every from-scratch run.

**Why this matters.** The learning rate is the single most important training dial: it scales how big a step the optimizer takes. Hold it fixed and you compromise. A rate that is safe early is too timid late; a rate that is fast late is reckless early. A schedule (a rule that changes the learning rate as a function of the step number) lets you have it both ways: cautious when the model is fragile, then bold, then careful again as it settles.

**The intuition.** Think of a sprinter. You do not explode off the line at full speed cold, you ease into it over the first few strides so you do not pull a muscle. That easing-in is warmup. Then you run hard. Then, as you approach the finish, you do not slam to a stop, you decelerate smoothly into a controlled finish. That smooth slowdown is the cosine decay. The whole schedule is: ramp up gently, cruise at the top, then glide down to nearly zero.

**The mechanics.** Early in training, AdamW's running moment estimates (from the last lesson) are still "cold", based on only a handful of gradients, so they are unreliable and a large step can destabilize everything. Warmup linearly raises the learning rate from near zero up to the base rate over the first `warmup` steps, buying time for the moments to warm up. After that, cosine decay follows the smooth shoulder of a cosine curve from the base rate down toward zero across the remaining steps. The tail-end tiny steps let the model anneal: make finer and finer adjustments and settle gently into a good minimum (a low point in the loss) instead of skidding past it. Here `cosine_lr(step, warmup=20, total=200, base_lr=3e-4)` returns the learning rate for any given step.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pocketlm import cosine_lr

total = 200
lrs = [cosine_lr(s, warmup=20, total=total, base_lr=3e-4) for s in range(total)]
plt.plot(lrs)
plt.xlabel("step")
plt.ylabel("learning rate")
plt.title("warmup + cosine decay")
plt.show()
print("start:", round(lrs[0], 7), " peak:", round(lrs[19], 7),
      " end:", round(lrs[-1], 8))

**What just happened.** The plot draws the learning rate against the step number, and you can read the story directly off the shape: a short straight ramp up to the peak around step 20, then a long graceful curve down toward zero. The printout confirms it numerically: the start value is tiny, the peak (step 19) is near the base rate of 3e-4, and the end value is close to zero. That single curve, ramp then glide, is the workhorse schedule behind a huge fraction of real training runs.

⚠️ **Common pitfalls**
- Skipping warmup entirely on a from-scratch model: the first few full-size steps hit while the moments are cold and can spike the loss or diverge.
- Making warmup absurdly long (say half of all steps): you waste most of training crawling at low rates and the model under-trains.
- Decaying all the way to exactly zero too early: the model stops learning before it is done. Decaying toward a small floor, not hard zero, is common.

🏋️ **Try it yourself.** Re-run with `warmup=0` and look at the very first learning-rate value: it jumps straight to the peak with no ramp. Then try `warmup=100` (out of 200 total) and see how the ramp eats half the schedule. Plot both alongside the original to feel how warmup trades early safety against time spent at full speed.

In [ ]:
assert lrs[0] < lrs[19]      # warming up
assert lrs[-1] < lrs[19]     # decaying